<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z325_Naive_RollingQuantiles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Naive de Rolling Quantiles

## La idea

Un modelo naive no aprende nada — usa directamente la historia reciente como predicción.
El **naive de rolling quantiles** toma el quantil de los últimos N meses:

```
pred_202002 = quantil(tn_{t-W:t}, q)
```

Ventajas sobre el promedio simple:
- La **mediana (q50)** no se arrastra por outliers — más robusta que la media
- **q75** captura una predicción optimista (si las ventas tienden a subir)
- **q25** captura una predicción conservadora

## ¿Por qué es útil compararlo?

Si la mediana de los últimos 6 meses da un score similar a AutoGluon (0.265) o HAR (0.269), entonces todos esos modelos complejos prácticamente no están mejorando sobre un naive. Es el **test de humildad** de cualquier modelo de forecasting.

## Variantes que probamos

| Variante | Ventana | Quantil | Intuición |
|---|---|---|---|
| mediana_6m | 6 meses | 0.50 | Centro del comportamiento reciente |
| mediana_12m | 12 meses | 0.50 | Centro del año completo |
| mediana_3m | 3 meses | 0.50 | Solo el trimestre más reciente |
| q75_6m | 6 meses | 0.75 | Optimista |
| q25_6m | 6 meses | 0.25 | Conservador |
| mismo_mes_año_anterior | 12 meses atrás | — | Naive estacional puro |
| promedio_12m | 12 meses | media | Baseline clásico (referencia) |

## 0.1 Init ambiente Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Setup

In [ ]:
!pip install uv
!uv pip install -q kaggle

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
  import os
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  os.system(comando)

In [ ]:
import os
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [ ]:
PARAM = {
  'experimento': 'NaiveRollingQ-01',
  'kaggle_competition': 'labo-iii-2026-rosario',
  # variante a submitear — cambiar y correr solo la seccion 4
  # opciones: 'mediana_6m', 'mediana_12m', 'mediana_3m',
  #           'q75_6m', 'q25_6m', 'mismo_mes', 'promedio_12m'
  'variante': 'mediana_6m'
}

In [ ]:
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

# 2  Preparacion de datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")

tb_ventas = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])
print(f"{tb_ventas['product_id'].n_unique()} productos")

# 3  Calculo de todas las variantes

Para cada producto calculamos las 7 variantes naive de una sola vez.
El ultimo mes conocido es `201912` — todas las ventanas se calculan hacia atrás desde ahí.

In [ ]:
productos = tb_apredecir["product_id"].to_list()
resultados = []

for pid in productos:
    serie = (
        tb_ventas.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )
    T = len(serie)

    # ventanas sobre los ultimos meses (hacia atras desde 201912)
    w3  = serie[max(0, T-3) : T]
    w6  = serie[max(0, T-6) : T]
    w12 = serie[max(0, T-12): T]

    # mismo mes del año anterior (201912 → 201812, posicion T-13)
    mismo_mes = serie[T - 13] if T >= 13 else w12.mean()

    resultados.append({
        'product_id':   pid,
        'mediana_3m':   float(np.median(w3)),
        'mediana_6m':   float(np.median(w6)),
        'mediana_12m':  float(np.median(w12)),
        'q75_6m':       float(np.quantile(w6, 0.75)),
        'q25_6m':       float(np.quantile(w6, 0.25)),
        'mismo_mes':    float(mismo_mes),
        'promedio_12m': float(w12.mean()),
    })

tb_naives = pl.DataFrame(resultados)
display(tb_naives.head(10))

## 3.1 Comparacion entre variantes

Visualizamos la distribucion de cada variante para entender qué tan distintas son entre sí.

In [ ]:
variantes = ['mediana_3m', 'mediana_6m', 'mediana_12m', 'q75_6m', 'q25_6m', 'mismo_mes', 'promedio_12m']

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for i, v in enumerate(variantes):
    vals = np.log1p(tb_naives[v].to_numpy())
    axes[i].hist(vals, bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
    axes[i].set_title(v, fontsize=9)
    axes[i].set_xlabel('log(1 + pred)', fontsize=8)

axes[-1].set_visible(False)
fig.suptitle('Distribucion de predicciones naive por variante (escala log)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# correlacion entre variantes — si son muy correlacionadas, son casi lo mismo
import pandas as pd
corr = tb_naives.select(variantes).to_pandas().corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, vmin=0.5, vmax=1.0, cmap='Blues')
plt.colorbar(im)
ax.set_xticks(range(len(variantes)))
ax.set_yticks(range(len(variantes)))
ax.set_xticklabels(variantes, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(variantes, fontsize=8)
for i in range(len(variantes)):
    for j in range(len(variantes)):
        ax.text(j, i, f"{corr.values[i,j]:.2f}", ha='center', va='center', fontsize=7)
ax.set_title('Correlacion entre variantes naive')
plt.tight_layout()
plt.show()

# 4  Submit a Kaggle

Cambiando `PARAM['variante']` al inicio y corriendo solo esta sección podés submitear cada variante sin recalcular nada.

In [ ]:
variante = PARAM['variante']

tb_final = tb_naives.select(['product_id', variante]).rename({variante: 'tn'})

# negativos a cero (no deberian haber pero por seguridad)
tb_final = tb_final.with_columns(
    pl.when(pl.col('tn') < 0).then(0.0).otherwise(pl.col('tn')).alias('tn')
)

display(tb_final)
print(f"Variante: {variante}  |  Nulls: {tb_final['tn'].is_null().sum()}")

In [ ]:
archivo = f"Naive_{variante}.csv"
mensaje = f"Naive rolling quantile {variante}"

tb_final.write_csv(archivo)
kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)

# 5  Tabla de resultados esperados

Completar a medida que vas submiteando:

| Variante | Score Kaggle | vs AutoGluon (0.265) |
|---|---|---|
| promedio_12m | | |
| mediana_3m | | |
| mediana_6m | | |
| mediana_12m | | |
| q75_6m | | |
| q25_6m | | |
| mismo_mes | | |

**Lectura clave**:
- Si algún naive llega a 0.265-0.270 → AutoGluon no está aportando casi nada sobre el nivel histórico
- Si los naives dan 0.30+ → los modelos complejos sí tienen valor real
- `mismo_mes` vs `mediana_12m` → ¿importa más la estacionalidad o el nivel promedio?